# Fake.br: detecção de anomalias com Isolation Forest

Notebook autossuficiente para Colab e execução local. O modelo aprende o padrão de notícias **True (0)** e sinaliza textos com comportamento linguístico atípico. Notícias **Fake (1)** ficam fora do ajuste e da calibração; são usadas somente para avaliação exploratória.

> **Interpretação:** `anomalyScore` é um score de anomalia, não uma probabilidade de falsidade. Um alerta indica necessidade de revisão e não comprova que a notícia seja falsa.

Para usar no Colab, faça upload deste arquivo e escolha **Runtime → Run all**. O ZIP do Fake.br é baixado automaticamente para `data/` quando ainda não estiver no ambiente.

## 1. Imports e configuração do ambiente

Execute esta célula em um kernel limpo. No Colab, as bibliotecas `numpy`, `pandas`, `scikit-learn` e `matplotlib` normalmente já estão disponíveis; se o ambiente solicitar, instale-as antes de executar todas as células.

In [ ]:
import hashlib
import platform
import re
import unicodedata
from importlib.metadata import version
from itertools import combinations
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline


In [ ]:
RANDOM_STATE = 42
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})

print("Python:", platform.python_version())
print({
    package_name: version(package_name)
    for package_name in ["numpy", "pandas", "scikit-learn", "matplotlib", "ipykernel"]
})


## 2. Carregamento dos dados

Mesmo ZIP e revisão fixa do notebook original. Textos completos e metadados são
carregados para auditoria; o truncamento acontece ANTES da extração usada nos
modelos. Nenhuma notícia é excluída por comprimento.

In [ ]:
CORPUS_REVISION = "780f5516c4ae070761632d98ac3368f3ded09d35"
CORPUS_URL = f"https://codeload.github.com/roneysco/Fake.br-Corpus/zip/{CORPUS_REVISION}"
project_folder = Path.cwd()
data_folder = project_folder / "data"
data_folder.mkdir(exist_ok=True)
archive_path = data_folder / f"Fake.br-Corpus-{CORPUS_REVISION}.zip"

if not archive_path.exists():
    temporary_path = archive_path.with_suffix(".download")
    with urlopen(CORPUS_URL, timeout=120) as response, temporary_path.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            target.write(chunk)

    with ZipFile(temporary_path) as archive:
        assert archive.testzip() is None, "ZIP corrompido; refaça o download."
    temporary_path.replace(archive_path)

print("Corpus revision:", CORPUS_REVISION)
print("Archive SHA256:", hashlib.sha256(archive_path.read_bytes()).hexdigest())


In [ ]:
metadata_columns = [
    "autor",
    "link",
    "categoria",
    "data_publicacao",
    "num_tokens",
    "num_palavras",
    "num_types",
    "num_links",
    "num_maiusculas",
    "num_verbos",
    "num_verbos_subj_imp",
    "num_substantivos",
    "num_adjetivos",
    "num_adverbios",
    "num_verbos_modais",
    "num_pron_1_2_sing",
    "num_pron_1_plural",
    "num_pronomes",
    "pausalidade",
    "num_caracteres",
    "tam_medio_sentenca",
    "tam_medio_palavra",
    "pct_erros_ortograficos",
    "emotividade",
    "diversidade",
]


In [ ]:
def load_news_texts(archive, folder, label):
    """Keep each source folder so the external label can be checked."""
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}/" not in name or not name.endswith(".txt"):
            continue

        article_id = Path(name).stem + ("t" if label == 0 else "")
        records.append({
            "id": article_id,
            "text": archive.read(name).decode("utf-8"),
            "label": label,
            "sourceClass": folder,
        })

    assert records, f"Nenhum texto encontrado em {folder}"
    return pd.DataFrame(records)


def parse_news_metadata(archive, name, label):
    """Build one metadata record and preserve the corpus schema check."""
    values = [line.strip() for line in archive.read(name).decode("utf-8").splitlines()]
    assert len(values) == len(metadata_columns), (
        f"Esquema inesperado em {name}: {len(values)} linhas"
    )
    article_id = Path(name).name.removesuffix("-meta.txt")
    article_id += "t" if label == 0 else ""
    return {
        **dict(zip(metadata_columns, values)),
        "id": article_id,
        "metadataLabel": label,
    }


def load_news_metadata(archive, folder, label):
    """Retain metadata so text-to-metadata pairing can be audited."""
    metadata_folder = f"/full_texts/{folder}-meta-information/"
    metadata_names = [
        name
        for name in sorted(archive.namelist())
        if metadata_folder in name and name.endswith("-meta.txt")
    ]
    records = [
        parse_news_metadata(archive, name, label)
        for name in metadata_names
    ]
    assert records, f"Nenhum metadado encontrado em {folder}"
    return pd.DataFrame(records)


In [ ]:
with ZipFile(archive_path) as archive:
    texts_frame = pd.concat(
        [
            load_news_texts(archive, "fake", 1),
            load_news_texts(archive, "true", 0),
        ],
        ignore_index=True,
    )
    metadata_frame = pd.concat(
        [
            load_news_metadata(archive, "fake", 1),
            load_news_metadata(archive, "true", 0),
        ],
        ignore_index=True,
    )

assert texts_frame["id"].is_unique and metadata_frame["id"].is_unique
assert set(texts_frame["id"]) == set(metadata_frame["id"]), (
    "Texto/metadados sem correspondência"
)
news_frame = texts_frame.merge(
    metadata_frame,
    on="id",
    validate="one_to_one",
    indicator=True,
)
assert news_frame["_merge"].eq("both").all()
assert news_frame["label"].eq(news_frame["metadataLabel"]).all()
news_frame = news_frame.drop(columns=["_merge", "metadataLabel"])
assert news_frame["text"].str.strip().ne("").all()
print(f"{len(news_frame):,} notícias carregadas; textos e metadados correspondem 1:1.")


## 3. Contrato dos rótulos

**0 = True; 1 = Fake.** O rótulo define as partições e permite avaliação externa.
Ele não entra como feature. A origem nas pastas também é verificada.


In [ ]:
label_names = {0: "True", 1: "Fake"}
assert label_names == {0: "True", 1: "Fake"}
assert set(news_frame["label"].unique()) == {0, 1}
assert news_frame.loc[news_frame["label"].eq(0), "sourceClass"].eq("true").all()
assert news_frame.loc[news_frame["label"].eq(1), "sourceClass"].eq("fake").all()

news_counts = news_frame.groupby(["label", "sourceClass"]).size()
display(news_counts.rename("newsCount").to_frame())


## 4. Preparação dos metadados

Preservamos todos os campos brutos em `raw_news_frame` e todas as contagens numéricas
em `news_frame`. Valores não numéricos viram NaN com contagem explícita; nenhum
ausente é preenchido antes do treino. `tem_autor` segue a lógica do original:
ausência para string vazia, `None`, `none` ou `NULL`. Não é uma medida de credibilidade.

In [ ]:
raw_news_frame = news_frame.copy(deep=True)
numeric_metadata_columns = metadata_columns[4:]
converted_metadata = news_frame[numeric_metadata_columns].apply(
    pd.to_numeric,
    errors="coerce",
)
conversion_missing = converted_metadata.isna().sum().rename(
    "missingAfterNumericConversion"
)
display(conversion_missing.to_frame())
news_frame[numeric_metadata_columns] = converted_metadata
author_values = (
    news_frame["autor"]
    .fillna("")
    .astype(str)
    .str.strip()
)
news_frame["tem_autor"] = (
    ~author_values.isin(["", "None", "none", "NULL"])
).astype(int)


## 5. Extração no texto efetivamente utilizado

Normalização Unicode NFKC, remoção do BOM inicial e primeiros CHARACTER_LIMIT
caracteres, incluindo espaços/pontuação. Textos menores permanecem menores, sem
preenchimento. O corte pode dividir palavras/frases. Palavras são sequências de
letras com hífen/apóstrofo interno; tokens também incluem números e pontuação.
Tipos são palavras distintas ignorando caixa. TTR = tipos/tokens; diversidade =
tipos/palavras. Maiúsculas conta palavras totalmente maiúsculas com mais de uma
letra. Links são URLs http(s) presentes no corpo. Estas definições explícitas
não pretendem reproduzir o extrator desconhecido dos metadados históricos.

As demais contagens linguísticas são marcadas ausentes na cópia de features,
não imputadas nem selecionadas pelo modelo; originais ficam em raw_news_frame e
news_frame. Autor permanece como metadado válido da notícia inteira.

In [ ]:
def calculate_ratio(numerator, denominator):
    """Avoid infinite densities when a text has no measurable denominator."""
    numerator = pd.to_numeric(numerator, errors="coerce").astype(float)
    denominator = pd.to_numeric(denominator, errors="coerce").astype(float)
    safe_denominator = denominator.where(
        denominator.gt(0) & np.isfinite(denominator)
    )
    return numerator.div(safe_denominator).replace([np.inf, -np.inf], np.nan)


In [ ]:
CHARACTER_LIMIT = 300
anomaly_columns = [
    "tem_autor",
    "typeTokenRatio",
    "linkDensity",
    "punctuationDensity",
    "uppercaseRatio",
    "diversidade",
]
word_pattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*", re.UNICODE)
token_pattern = re.compile(
    r"[^\W\d_]+(?:['’\-][^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]",
    re.UNICODE,
)


def count_text_features(normalized_text):
    """Count the lexical patterns retained in the measured text prefix."""
    words = word_pattern.findall(normalized_text)
    return {
        "num_palavras": len(words),
        "num_tokens": len(token_pattern.findall(normalized_text)),
        "num_types": len({word.casefold() for word in words}),
        "num_links": len(
            re.findall(r"https?://\S+", normalized_text, flags=re.IGNORECASE)
        ),
        "num_maiusculas": sum(
            word.isupper() and len(word) > 1 for word in words
        ),
    }


def measure_text(text, character_limit):
    """Measure the text prefix used by the unsupervised model."""
    normalized_text = unicodedata.normalize("NFKC", text).lstrip("\ufeff")
    if character_limit is not None and (
        not isinstance(character_limit, int) or character_limit < 1
    ):
        raise ValueError("Limite deve ser inteiro positivo ou None.")
    if character_limit is not None:
        normalized_text = normalized_text[:character_limit]

    return {
        "text": normalized_text,
        **count_text_features(normalized_text),
        "num_caracteres": len(normalized_text),
    }


def add_text_measurements(features_frame, news_frame, character_limit):
    measurements = pd.DataFrame(
        [measure_text(text, character_limit) for text in news_frame["text"]],
        index=news_frame.index,
    )
    for column in measurements:
        features_frame[column] = measurements[column]
    return features_frame


def add_derived_features(features_frame):
    ratio_inputs = {
        "typeTokenRatio": ("num_types", "num_tokens"),
        "diversidade": ("num_types", "num_palavras"),
        "linkDensity": ("num_links", "num_palavras"),
        "uppercaseRatio": ("num_maiusculas", "num_palavras"),
    }
    for feature_name, (numerator_column, denominator_column) in ratio_inputs.items():
        features_frame[feature_name] = calculate_ratio(
            features_frame[numerator_column],
            features_frame[denominator_column],
        )

    features_frame["punctuationDensity"] = calculate_ratio(
        features_frame["num_tokens"] - features_frame["num_palavras"],
        features_frame["num_tokens"],
    )
    return features_frame


def extract_anomaly_features(news_frame, character_limit=CHARACTER_LIMIT):
    """Preserve raw data while adding the exact text features used for modeling."""
    features_frame = news_frame.copy(deep=True)
    features_frame["num_palavras_original"] = news_frame["num_palavras"]
    features_frame["text_original"] = news_frame["text"]
    features_frame[numeric_metadata_columns] = np.nan
    features_frame = add_text_measurements(
        features_frame,
        news_frame,
        character_limit,
    )
    return add_derived_features(features_frame)


In [ ]:
features_frame = extract_anomaly_features(news_frame)
features_frame[anomaly_columns] = features_frame[anomaly_columns].replace(
    [np.inf, -np.inf],
    np.nan,
)
missing_counts = features_frame[anomaly_columns].isna().sum()
display(missing_counts.rename("missingCount").to_frame())
print("Limite de caracteres:", CHARACTER_LIMIT)
display(
    features_frame.groupby("label")[[
        "num_palavras_original",
        "num_palavras",
        "num_caracteres",
    ]].agg(["min", "median", "max"])
)
print(
    "Textos menores que o limite:",
    int(features_frame["num_caracteres"].lt(CHARACTER_LIMIT).sum()),
)


## 6. Verificações de integridade

As seis features usam o prefixo do texto; a presença de autor vem dos metadados.
As contagens originais ficam preservadas para auditoria e não alimentam o modelo.


In [ ]:
assert "num_palavras" not in anomaly_columns
assert "label" not in anomaly_columns and "id" not in anomaly_columns
assert len(anomaly_columns) == len(set(anomaly_columns)) == 6
assert features_frame["id"].is_unique
assert features_frame["num_caracteres"].le(CHARACTER_LIMIT).all()
assert features_frame["num_palavras_original"].equals(news_frame["num_palavras"])
assert features_frame["num_verbos"].isna().all()
assert not np.isinf(features_frame[anomaly_columns].to_numpy(dtype=float)).any()
assert measure_text("casa " * 100, 300)["num_palavras"] == 60
assert measure_text("casa " * 100, 300)["num_caracteres"] == 300
print("Contagens do prefixo verificadas; num_palavras não entra no treinamento.")

## 7. Análise estatística

Estatísticas descritivas por rótulo, sem substituir os vetores individuais por médias.
Esta inspeção do corpus completo é exploratória: seus resultados não selecionam
features, hiperparâmetros nem o threshold. Qualquer escolha futura guiada por estes
resultados exige uma nova avaliação independente.

As correlações de Pearson com comprimento são mostradas no total e por classe para
evitar confundir efeitos de classe e de tamanho. Uma correlação NaN pode indicar
uma feature constante. Mantemos as seis features recalculadas, inclusive possíveis
redundâncias.


In [ ]:
def summarize_feature(label, label_name, values):
    first_quartile, third_quartile = values.quantile([0.25, 0.75])
    return {
        "label": label,
        "class": label_name,
        "feature": values.name,
        "count": values.count(),
        "mean": values.mean(),
        "median": values.median(),
        "std": values.std(),
        "min": values.min(),
        "Q1": first_quartile,
        "Q3": third_quartile,
        "IQR": third_quartile - first_quartile,
        "max": values.max(),
        "missingCount": values.isna().sum(),
    }


def build_feature_statistics(features_frame, feature_columns, label_names):
    """Summarize distributions without replacing individual feature vectors."""
    statistics_records = [
        summarize_feature(label, label_names[label], group[feature])
        for label, group in features_frame.groupby("label", sort=True)
        for feature in feature_columns
    ]
    return pd.DataFrame(statistics_records).set_index(
        ["label", "class", "feature"]
    )


feature_statistics = build_feature_statistics(
    features_frame,
    anomaly_columns,
    label_names,
)
display(feature_statistics)

length_columns = ["num_palavras", "num_tokens"]
correlation_groups = [
    ("All", features_frame),
    ("True (0)", features_frame.loc[features_frame["label"].eq(0)]),
    ("Fake (1)", features_frame.loc[features_frame["label"].eq(1)]),
]
length_correlations = pd.concat(
    {
        name: group[anomaly_columns + length_columns]
        .corr()
        .loc[anomaly_columns, length_columns]
        for name, group in correlation_groups
    },
    names=["group", "feature"],
)
display(length_correlations)

feature_correlations = features_frame[anomaly_columns].corr()
display(feature_correlations.round(3))
display(length_correlations.xs("typeTokenRatio", level="feature"))
print(
    "Pearson TTR vs diversidade:",
    feature_correlations.loc["typeTokenRatio", "diversidade"],
)


## 8. Partições de treino, validação e teste

True: 60% treino, 20% validação, 20% teste. Fake: 50% validação, 50% teste.
Todas as divisões usam `RANDOM_STATE = 42`. Verificamos IDs exclusivos e a cobertura
integral do corpus. **Nenhuma notícia Fake participa do ajuste ou da calibração.**

Limitação do protocolo: a divisão é por notícia, não por assunto, fonte ou data.
O corpus possui pares True/Fake com o mesmo número-base; os IDs `123t` e `123` são
notícias distintas, mas podem tratar do mesmo assunto em partições diferentes.
IDs exclusivos não demonstram independência temática.


In [ ]:
def validate_partitions(partitions, features_frame):
    """Ensure each article appears in exactly one complete data partition."""
    for name, frame in partitions.items():
        expected_label = 0 if name.startswith("normal") else 1
        assert not frame.empty and frame["id"].is_unique
        assert frame["label"].eq(expected_label).all()

    for (left_name, left), (right_name, right) in combinations(partitions.items(), 2):
        assert set(left["id"]).isdisjoint(right["id"]), (
            f"IDs compartilhados: {left_name}/{right_name}"
        )

    partition_ids = set().union(*(set(frame["id"]) for frame in partitions.values()))
    assert partition_ids == set(features_frame["id"])
    assert sum(len(frame) for frame in partitions.values()) == len(features_frame)


normal_frame = features_frame.loc[features_frame["label"].eq(0)].copy()
fake_frame = features_frame.loc[features_frame["label"].eq(1)].copy()
normal_train_frame, normal_holdout_frame = train_test_split(
    normal_frame,
    train_size=0.60,
    random_state=RANDOM_STATE,
)
normal_validation_frame, normal_test_frame = train_test_split(
    normal_holdout_frame,
    test_size=0.50,
    random_state=RANDOM_STATE,
)
fake_validation_frame, fake_test_frame = train_test_split(
    fake_frame,
    test_size=0.50,
    random_state=RANDOM_STATE,
)
partitions = {
    "normalTrain": normal_train_frame,
    "normalValidation": normal_validation_frame,
    "normalTest": normal_test_frame,
    "fakeValidation": fake_validation_frame,
    "fakeTest": fake_test_frame,
}
validate_partitions(partitions, features_frame)
assert normal_train_frame["label"].eq(0).all()
partition_summary = pd.DataFrame([
    {
        "partition": name,
        "count": len(frame),
        "label": int(frame["label"].iloc[0]),
    }
    for name, frame in partitions.items()
]).set_index("partition")
display(partition_summary)


## 9. Isolation Forest

Pipeline fixo: imputação pela mediana + 300 árvores, sem `StandardScaler`.
O único `.fit()` recebe exclusivamente `normal_train`. Uma guarda rejeita notícias
Fake, infinitos e features inteiramente ausentes no treino, o que evita a remoção
silenciosa de colunas pelo imputer. Nem médias do corpus nem dados de validação ou
teste participam do ajuste.


In [ ]:
def fit_normal_only(pipeline, training_frame):
    """Prevent Fake examples from entering the unsupervised model fit."""
    if training_frame.empty or not training_frame["label"].eq(0).all():
        raise ValueError("Treino permitido apenas com notícias True (label == 0).")

    training_features = training_frame[anomaly_columns]
    if np.isinf(training_features.to_numpy(dtype=float)).any():
        raise ValueError("Features de treino contêm infinitos.")

    empty_columns = training_features.columns[
        training_features.isna().all()
    ].tolist()
    if empty_columns:
        raise ValueError(
            f"Features inteiramente ausentes em normalTrain: {empty_columns}"
        )
    return pipeline.fit(training_features)


In [ ]:
anomaly_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median"),
    ),
    (
        "detector",
        IsolationForest(
            n_estimators=300,
            contamination="auto",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
])
fit_normal_only(anomaly_pipeline, normal_train_frame)
assert list(anomaly_pipeline.feature_names_in_) == anomaly_columns
assert anomaly_pipeline["detector"].n_features_in_ == len(anomaly_columns)
np.testing.assert_allclose(
    anomaly_pipeline["imputer"].statistics_,
    normal_train_frame[anomaly_columns].median().to_numpy(),
)
print(f"Pipeline ajustado em {len(normal_train_frame)} notícias True e 0 Fake.")
display(
    pd.Series(
        anomaly_pipeline["imputer"].statistics_,
        index=anomaly_columns,
        name="normalTrainMedian",
    ).to_frame()
)


## 10. Scores de anomalia

Invertimos `decision_function`: score menor = mais normal; maior = mais anômalo.
Os scores podem ser negativos. Não são probabilidades nem percentuais de falsidade.
A validação Fake é reservada para inspeção visual.


In [ ]:
def calculate_anomaly_score(pipeline, features_frame):
    """Orient scores so larger values consistently indicate stronger anomalies."""
    return -pipeline.decision_function(features_frame)


In [ ]:
normal_validation_scores = calculate_anomaly_score(
    anomaly_pipeline,
    normal_validation_frame[anomaly_columns],
)
fake_validation_scores = calculate_anomaly_score(
    anomaly_pipeline,
    fake_validation_frame[anomaly_columns],
)
test_frame = pd.concat(
    [normal_test_frame, fake_test_frame],
    ignore_index=True,
)
test_results_frame = test_frame[["id", "label"] + anomaly_columns].copy()
test_results_frame["anomalyScore"] = calculate_anomaly_score(
    anomaly_pipeline,
    test_frame[anomaly_columns],
)
assert np.isfinite(normal_validation_scores).all()
assert np.isfinite(fake_validation_scores).all()
assert np.isfinite(test_results_frame["anomalyScore"]).all()


## 11. Limiar

Regra fixa: **percentil 95 dos scores de `normal_validation`**. Aproximadamente 5%
das notícias True de validação ultrapassam esse valor; empates podem aumentar
a proporção porque usamos `>=`. A taxa no teste pode ser diferente. Nenhuma notícia
Fake ou de teste define o limiar; `contamination="auto"` não escolhe este corte.
O notebook não usa `predict()` do detector.


In [ ]:
threshold = np.quantile(normal_validation_scores, 0.95)
test_results_frame["isAnomaly"] = (
    test_results_frame["anomalyScore"] >= threshold
)
normal_validation_anomaly_rate = np.mean(normal_validation_scores >= threshold)
print(f"Threshold (percentil 95 de normalValidation): {threshold:.6f}")
print(
    "True de validação marcadas como anômalas:",
    f"{normal_validation_anomaly_rate:.2%}",
)
assert np.isfinite(threshold)
assert "isFake" not in test_results_frame.columns
display(
    test_results_frame[["id", "label", "anomalyScore", "isAnomaly"]].head()
)


## 12. Avaliação

A avaliação final usa `normal_test + fake_test`. A classe positiva é **Fake (1)**
somente para medir discriminação. Precision, Recall, F1 e a matriz comparam o rótulo
externo com o sinal `isAnomaly`, sem tratá-lo como classificação factual.
ROC-AUC e Average Precision usam scores contínuos. Reportamos **Average Precision
(AP)** como resumo da curva PR, não a integral trapezoidal PR-AUC.

A prevalência Fake no teste é a referência de AP para uma ordenação aleatória;
ela não representa a prevalência real na web.


In [ ]:
score_statistics = test_results_frame.groupby("label")["anomalyScore"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)
score_statistics.insert(0, "class", score_statistics.index.map(label_names))
display(score_statistics)

test_labels = test_results_frame["label"].to_numpy()
test_scores = test_results_frame["anomalyScore"].to_numpy()
test_flags = test_results_frame["isAnomaly"].to_numpy(dtype=int)
confusion_matrix_values = confusion_matrix(test_labels, test_flags, labels=[0, 1])
true_negative, false_positive, false_negative, true_positive = (
    confusion_matrix_values.ravel()
)
false_positive_rate = false_positive / (true_negative + false_positive)
fake_detection_rate = true_positive / (false_negative + true_positive)
fake_prevalence = np.mean(test_labels == 1)
metrics = {
    "ROC-AUC": roc_auc_score(test_labels, test_scores),
    "Average Precision (AP)": average_precision_score(test_labels, test_scores),
    "Precision": precision_score(test_labels, test_flags, zero_division=0),
    "Recall": recall_score(test_labels, test_flags, zero_division=0),
    "F1": f1_score(test_labels, test_flags, zero_division=0),
    "True false positive rate": false_positive_rate,
    "Fake detection rate": fake_detection_rate,
    "Fake prevalence (AP baseline)": fake_prevalence,
}
display(pd.Series(metrics, name="value").to_frame())
print(f"Notícias Fake detectadas como anômalas: {fake_detection_rate:.2%}")
print(f"Notícias True marcadas como anômalas (FPR): {false_positive_rate:.2%}")
display(pd.DataFrame(
    confusion_matrix_values,
    index=["Actual True (0)", "Actual Fake (1)"],
    columns=["Within threshold", "Anomaly"],
))
assert confusion_matrix_values.sum() == len(test_results_frame)
assert set(test_results_frame["id"]) == (
    set(normal_test_frame["id"]) | set(fake_test_frame["id"])
)
assert np.isclose(fake_detection_rate, metrics["Recall"])


## 13. Visualizações

True (0) aparece em azul e Fake (1) em laranja. Os histogramas de densidade usam
intervalos comuns porque o teste tem tamanhos de classe diferentes. O primeiro
painel mostra a validação apenas para inspeção; os demais mostram o teste.
O limiar é o mesmo, escolhido exclusivamente em `normal_validation`.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
true_scores = test_results_frame.loc[
    test_results_frame["label"].eq(0), "anomalyScore"
]
fake_scores = test_results_frame.loc[
    test_results_frame["label"].eq(1), "anomalyScore"
]
score_panels = [
    (
        axes[0, 0],
        [normal_validation_scores, fake_validation_scores],
        "Validação: inspeção dos scores",
    ),
    (
        axes[0, 1],
        [true_scores, fake_scores],
        "Teste: distribuição dos scores",
    ),
]
for axis, groups, title in score_panels:
    bins = np.histogram_bin_edges(np.concatenate(groups), bins=40)
    for values, name, color in zip(
        groups,
        ["True (0)", "Fake (1)"],
        ["tab:blue", "tab:orange"],
    ):
        axis.hist(
            values,
            bins=bins,
            density=True,
            alpha=0.5,
            label=name,
            color=color,
        )
    axis.axvline(
        threshold,
        color="black",
        linestyle="--",
        label=f"Threshold = {threshold:.3f}",
    )
    axis.set(
        title=title,
        xlabel="anomalyScore (maior = mais anômalo)",
        ylabel="Densidade",
    )
    axis.legend(fontsize=8)

boxes = axes[0, 2].boxplot([true_scores, fake_scores], patch_artist=True)
for box, color in zip(boxes["boxes"], ["tab:blue", "tab:orange"]):
    box.set_facecolor(color)
    box.set_alpha(0.5)
axes[0, 2].set_xticks([1, 2], ["True (0)", "Fake (1)"])
axes[0, 2].axhline(
    threshold,
    color="black",
    linestyle="--",
    label="Threshold",
)
axes[0, 2].set(title="Teste: boxplot por label", ylabel="anomalyScore")
axes[0, 2].legend(fontsize=8)

roc_fpr, roc_tpr, _ = roc_curve(test_labels, test_scores, pos_label=1)
axes[1, 0].plot(
    roc_fpr,
    roc_tpr,
    label=f"ROC-AUC = {metrics['ROC-AUC']:.3f}",
)
axes[1, 0].plot([0, 1], [0, 1], "k--", label="Referência aleatória")
axes[1, 0].scatter(
    [false_positive_rate],
    [fake_detection_rate],
    color="black",
    label="Threshold fixo",
)
axes[1, 0].set(
    title="Teste: ROC (positivo = Fake)",
    xlabel="False positive rate (True)",
    ylabel="True positive rate (Fake)",
)
axes[1, 0].legend(fontsize=8)

pr_precision, pr_recall, _ = precision_recall_curve(
    test_labels,
    test_scores,
    pos_label=1,
)
axes[1, 1].plot(
    pr_recall,
    pr_precision,
    label=f"AP = {metrics['Average Precision (AP)']:.3f}",
)
axes[1, 1].axhline(
    fake_prevalence,
    color="black",
    linestyle="--",
    label=f"Prevalência Fake = {fake_prevalence:.3f}",
)
axes[1, 1].scatter(
    [metrics["Recall"]],
    [metrics["Precision"]],
    color="black",
    label="Threshold fixo",
)
axes[1, 1].set(
    title="Teste: Precision-Recall (positivo = Fake)",
    xlabel="Recall",
    ylabel="Precision",
    ylim=(0, 1.05),
)
axes[1, 1].legend(fontsize=8)

ConfusionMatrixDisplay(
    confusion_matrix_values,
    display_labels=["True (0)", "Fake (1)"],
).plot(ax=axes[1, 2], colorbar=False, cmap="Blues", values_format="d")
axes[1, 2].set_xticks([0, 1], ["Dentro do padrão", "Anomalia"])
axes[1, 2].set(
    title="Teste: matriz após threshold",
    xlabel="Sinal do detector",
    ylabel="Rótulo real",
)
axes[1, 2].grid(False)
plt.show()


## 14. Casos extremos

Os exemplos vêm apenas do teste: cinco True com maior score, cinco Fake com maior
score e cinco Fake com menor score. As seis features são apresentadas; não são
atribuições causais nem importâncias do Isolation Forest. Um NaN na tabela indica
que a feature estava ausente antes da imputação e foi preservada para auditoria.


In [ ]:
example_columns = ["id", "label", "anomalyScore", "isAnomaly"] + anomaly_columns
extreme_cases = {
    "5 True com maior anomalyScore": test_results_frame.loc[
        test_results_frame["label"].eq(0)
    ].nlargest(5, "anomalyScore"),
    "5 Fake com maior anomalyScore": test_results_frame.loc[
        test_results_frame["label"].eq(1)
    ].nlargest(5, "anomalyScore"),
    "5 Fake com menor anomalyScore": test_results_frame.loc[
        test_results_frame["label"].eq(1)
    ].nsmallest(5, "anomalyScore"),
}
for title, examples in extreme_cases.items():
    display(Markdown(f"**{title}**"))
    display(examples[example_columns].reset_index(drop=True))


## 15. Conclusão

O resumo abaixo é calculado nesta execução, sem ajuste posterior do modelo ou do
limiar. AUC resume a ordenação; recall e FPR resumem o corte escolhido. Nenhuma
dessas métricas demonstra significância estatística ou validade factual.


In [ ]:
display(Markdown(f"""
Foram treinadas **{len(normal_train_frame)} notícias True e nenhuma Fake**,
usando até **{CHARACTER_LIMIT} caracteres** e seis features recalculadas.
No teste: **{true_positive}/{len(fake_test_frame)} Fake detectadas** e
**{false_positive}/{len(normal_test_frame)} True sinalizadas**.
ROC-AUC: **{metrics['ROC-AUC']:.4f}**; recall **{fake_detection_rate:.2%}**;
FPR **{false_positive_rate:.2%}**. Corte: **{threshold:.6f}**, calibrado apenas em True.

num_palavras permanece na extração, mas está fora da matriz de treino.
Correlações por classe estão na análise estatística. Caracteres fixos não fixam palavras;
o corte também remove conteúdo e pode cortar frases. Não ajustamos o corte pelo teste.
A comparação com o texto completo não faz parte deste notebook didático.
**Anomalia não comprova falsidade.**
"""))
